<a href="https://colab.research.google.com/github/Akhila-010/GenAI_Tasks/blob/main/Mini_ChatGPT.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q openai faiss-cpu PyPDF2 tiktoken markdown

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 34.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 232.6/232.6 kB 7.1 MB/s eta 0:00:00


In [ ]:
import os
import faiss
import numpy as np
from openai import OpenAI
from PyPDF2 import PdfReader
from google.colab import files
from getpass import getpass

In [ ]:
api_key = getpass("Enter your OpenAI API Key: ")

client = OpenAI(api_key=api_key)

Enter your OpenAI API Key: ··········


In [ ]:
uploaded = files.upload()

Saving information-brochure_Delhi.pdf to information-brochure_Delhi.pdf


In [ ]:
documents = []

for filename in uploaded.keys():

    print(f"\nProcessing file: {filename}")

    # ---------- PDF ----------
    if filename.endswith(".pdf"):

        text = ""

        reader = PdfReader(filename)

        for page in reader.pages:
            extracted = page.extract_text()

            if extracted:
                text += extracted + "\n"

        documents.append(text)

    # ---------- TXT ----------
    elif filename.endswith(".txt"):

        with open(filename, "r", encoding="utf-8") as f:
            text = f.read()

        documents.append(text)

    # ---------- MARKDOWN ----------
    elif filename.endswith(".md"):

        with open(filename, "r", encoding="utf-8") as f:
            text = f.read()

        documents.append(text)

print("\nDocuments Loaded Successfully!")
print("Total Documents:", len(documents))


Processing file: information-brochure_Delhi.pdf

Documents Loaded Successfully!
Total Documents: 1


In [ ]:
def chunk_text(text, chunk_size=500, overlap=50):

    chunks = []

    start = 0

    while start < len(text):

        end = start + chunk_size

        chunk = text[start:end]

        chunks.append(chunk)

        start += chunk_size - overlap

    return chunks

In [ ]:

all_chunks = []

for doc in documents:

    chunks = chunk_text(doc)

    all_chunks.extend(chunks)

print("\nTotal Chunks Created:", len(all_chunks))



Total Chunks Created: 319


In [ ]:
#Generate Embeddings
def get_embedding(text):

    response = client.embeddings.create(
        model="text-embedding-3-small",
        input=text
    )

    embedding = response.data[0].embedding

    return embedding


print("\nGenerating embeddings...")

embeddings = []

for i, chunk in enumerate(all_chunks):

    embedding = get_embedding(chunk)

    embeddings.append(embedding)

    print(f"Processed Chunk {i+1}/{len(all_chunks)}")

print("\nEmbeddings Generated Successfully!")


Generating embeddings...
Processed Chunk 1/319
Processed Chunk 2/319
Processed Chunk 3/319
Processed Chunk 4/319
Processed Chunk 5/319
Processed Chunk 6/319
Processed Chunk 7/319
Processed Chunk 8/319
Processed Chunk 9/319
Processed Chunk 10/319
Processed Chunk 11/319
Processed Chunk 12/319
Processed Chunk 13/319
Processed Chunk 14/319
Processed Chunk 15/319
Processed Chunk 16/319
Processed Chunk 17/319
Processed Chunk 18/319
Processed Chunk 19/319
Processed Chunk 20/319
Processed Chunk 21/319
Processed Chunk 22/319
Processed Chunk 23/319
Processed Chunk 24/319
Processed Chunk 25/319
Processed Chunk 26/319
Processed Chunk 27/319
Processed Chunk 28/319
Processed Chunk 29/319
Processed Chunk 30/319
Processed Chunk 31/319
Processed Chunk 32/319
Processed Chunk 33/319
Processed Chunk 34/319
Processed Chunk 35/319
Processed Chunk 36/319
Processed Chunk 37/319
Processed Chunk 38/319
Processed Chunk 39/319
Processed Chunk 40/319
Processed Chunk 41/319
Processed Chunk 42/319
Processed Chunk 4

In [ ]:
#Create FAISS Vector Database
embedding_dimension = len(embeddings[0])

index = faiss.IndexFlatL2(embedding_dimension)

embedding_array = np.array(embeddings).astype("float32")

index.add(embedding_array)

print("\nFAISS Vector Database Created!")
print("Total vectors stored:", index.ntotal)


FAISS Vector Database Created!
Total vectors stored: 319


In [ ]:
#Search Function
def search_documents(question, top_k=3):

    query_embedding = get_embedding(question)

    query_array = np.array([query_embedding]).astype("float32")

    distances, indices = index.search(query_array, top_k)

    relevant_chunks = []

    for idx in indices[0]:

        relevant_chunks.append(all_chunks[idx])

    return relevant_chunks


In [ ]:

#Generate Final Answer

SYSTEM_PROMPT = """
You are a helpful AI Knowledge Assistant.

Answer the user's question ONLY from the provided context.

If the answer is not found in the context,
say:
'I could not find the answer in the uploaded documents.'
"""


def generate_answer(question, context):

    prompt = f"""
Context:
{context}

Question:
{question}
"""

    response = client.chat.completions.create(

        model="gpt-4.1-mini",

        messages=[
            {
                "role": "system",
                "content": SYSTEM_PROMPT
            },
            {
                "role": "user",
                "content": prompt
            }
        ],

        temperature=0.2
    )

    answer = response.choices[0].message.content

    return answer


In [ ]:
#Chat Loop

print("\n=================================")
print("KNOWLEDGE ASSISTANT READY!")
print("Type 'exit' to stop.")
print("=================================")

while True:

    question = input("\nAsk a Question: ")

    if question.lower() == "exit":
        print("Goodbye!")
        break

    # Retrieve relevant chunks
    relevant_chunks = search_documents(question)

    # Combine context
    context = "\n\n".join(relevant_chunks)

    # Generate answer
    answer = generate_answer(question, context)

    print("\nAnswer:")
    print(answer)


KNOWLEDGE ASSISTANT READY!
Type 'exit' to stop.

Ask a Question: is hostel facility is avaliable ?

Answer:
Yes, hostel facility is available for all full-time post-graduate students, subject to availability and the hostel accommodation policy. However, there is currently a severe shortage of hostel accommodation due to renovation activities and an increase in student strength, so the Institute may not be able to offer hostel accommodation to all post-graduate students.

Ask a Question: how many departments are there ?

Answer:
I could not find the answer in the uploaded documents.

Ask a Question: list the departments which are avaliable 

Answer:
The available departments are:

1. Engineering  
2. Humanities and Social Sciences  
3. Management Studies  
4. Mathematics  
5. Materials Science and Engineering  
6. Mechanical Engineering  
7. Physics  
8. Textile and Fibre Engineering

Ask a Question: exit
Goodbye!
